# The Sources

https://medium.com/@azimkhan8018/a-beginners-guide-to-deep-learning-with-mnist-dataset-0894f7183344 - syntax from here

https://www.tensorflow.org/datasets/keras_example - layer details from here

https://huggingface.co/docs/optimum/concept_guides/quantization - quantization from here

# Import modules...

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,Flatten
import matplotlib.pyplot as plt
import numpy as np
import os

# Looking at the training and test data...

In [ ]:
# This downloads the mnist dataset from keras
(x_train,y_train),(x_test,y_test) = keras.datasets.mnist.load_data()

print(y_train[67]) # Look, for it is the index 67 entry, and its label (y_train) is 1;
plt.figure()
plt.imshow(x_train[67]) # And the actual image plotted, is indeed 1! (A questionable-looking 1, but it's what it is)

In [ ]:
#Meanwhile the test data for testing accuracy, not training
print(f"Below is the number... {y_test[67]}")
plt.imshow(x_test[67])

#Same thing; (x) data <-> (y) label

# Preprocessing/normalization

In [ ]:
# The pixel range is between 0 to 255... normalize
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Constructing model

In [ ]:
print("As in the tensorflow docs example...")

model= Sequential()
model.add(Flatten(input_shape=(28,28))) # Since the image is already flat in memory, this isn't really an actual operation in the accelerator
model.add(Dense(128, activation='relu'))
model.add(Dense(10))
model.summary()

# Compile and train model!

In [ ]:
#Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

#Train
history = model.fit(
    x_train,
    y_train,
    batch_size=128,
    epochs=12,
    validation_split=0.2, #Uses some of the training data as a validation set
    shuffle=True,
)

# Evaluate model performance

In [ ]:
y_logits = model.predict(x_test)
y_pred = y_logits.argmax(axis=1)

from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, y_pred))

# Plot accuracy over epochs


In [ ]:
plt.plot(history.history["sparse_categorical_accuracy"], label="Training")
plt.plot(history.history["val_sparse_categorical_accuracy"], label="Validation")
plt.legend()

# Plot loss over epochs


In [ ]:
plt.plot(history.history['loss'], label="Training")
plt.plot(history.history['val_loss'], label="Validation")
plt.legend()

# Calibration

Here we find the scaling factor assuming...

signed INT8, zero-point 0, one scale per tensor.

In [ ]:
def scale(x):
  # Returns largest magnitude divided by 127
  # We divide the output of this to the original output to have it be
  # scaled such that its max is 127
  peak = float(np.max(np.abs(x)))
  if peak==0:
    return 1.0
  else:
    return peak/127

x = x_train[:48000]  # Calibration images exclude validation set
s_in = scale(x) # scale input
s_input_initial = s_in #for quantization

#multipliers=[]
layer_params = []

for layer in model.layers:
  x = layer(x) # Runs forward pass on input tensor layer with current layer
                 # x is now output

  if isinstance(layer, Dense):
    s_w = scale(layer.kernel.numpy()) # scale weights
    s_out = scale(x.numpy()) # scale outputs
    s_bias = s_in*s_w #for quantization
    multiplier = s_bias/s_out #for quantization

    #to keep for latter cells
    layer_params.append({
        "name": layer.name,
        "s_w": s_w,
        "s_bias": s_bias,
        "multiplier": multiplier,
        "has_relu": (layer.activation == keras.activations.relu),
    })

    print(f"{layer.name}:")
    print(f"  Weight scale:      {s_w}")
    print(f"  Output scale:      {s_out}")
    print(f"  Bias scale:        {s_bias}")
    print(f"  Output multiplier: {multiplier}")

    #multipliers.append(s_in*s_w/s_out)

    s_in = s_out

# Converting scaling factor per layer to shift and multiply

We convert the output multiplier per layer to a right shift (r) and unsigned multiply (M). Simply:

output multiplier ~ M/2^r

In [ ]:
def multiplier_and_shift(m):
  for shift in range(31, -1, -1): # largest shift gives highest precision
    multiplier = int(round(m * (1 << shift)))

    if 1 <= multiplier <= 0xFFFFFFFF: # max 32 bit multiplier
        return multiplier, shift

  raise ValueError("Multiplier is too large")

#quantized_multipliers=[]
#quantized_shifts=[]

#for multiplier in multipliers:
#  m, s = multiplier_and_shift(multiplier)
#  quantized_multipliers.append(m)
#  quantized_shifts.append(s)

#print(quantized_multipliers)
#print(quantized_shifts)

#just changed it to match yung quantization
for p in layer_params:
  m, s = multiplier_and_shift(p["multiplier"])
  p["q_multiplier"] = m
  p["q_shift"] = s
  print(f"{p['name']}: M={m}, shift={s}")


# Quantization and export

We quantize the weights to be signed INT8, and bias to be signed INT32 and export them.

In [ ]:
export_dict = {"s_input": np.array(s_input_initial, dtype=np.float32)}
dense_layers = [l for l in model.layers if isinstance(l, Dense)]

# for layer in model.layers:
#  if isinstance(layer, Dense):
#      weights, biases = layer.get_weights()
for layer, p in zip(dense_layers, layer_params):
  weights, biases = layer.get_weights()
  #symmetric int8 [-127, 127] with Z = 0
  q_weights = np.clip(np.round(weights/p["s_w"]), -127, 127).astype(np.int8)
  #int32 accumulation scale = s_in*s_w
  q_biases = np.round(biases/p["s_bias"]).astype(np.int32)

  p["q_weights"] = q_weights
  p["q_biases"] = q_biases

  export_dict[f"{p['name']}_weights"] = q_weights
  export_dict[f"{p['name']}_biases"] = q_biases
  export_dict[f"{p['name']}_multiplier"] = np.array(p["q_multiplier"], dtype=np.uint32)
  export_dict[f"{p['name']}_shift"] = np.array(p["q_shift"], dtype=np.int32)

  print(f"{p['name']}: weights={q_weights.shape} ({q_weights.dtype}), biases={q_biases.shape} ({q_biases.dtype})")

#export quantized model for python just in case
np.savez("mnist_quantized_int8.npz", **export_dict)
print("\nExported quantized parameters to 'mnist_quantized_int8.npz'\n")

#verify integer-only inference accuracy on x_test
q_x = np.clip(np.round(x_test.reshape(-1, 784)/s_input_initial), -127, 127).astype(np.int8)

for i, p in enumerate(layer_params):
  acc = np.matmul(q_x.astype(np.int32), p["q_weights"].astype(np.int32)) + p["q_biases"]
  # Final layer: keep biased INT32 scores, without output quantization
  if i == len(layer_params) - 1:
    q_x = acc
    break

  shift = p["q_shift"]
  mult = np.int64(p["q_multiplier"])
  product = acc.astype(np.int64) * mult

  if shift > 0:
    magnitude = (np.abs(product) + (1 << (shift - 1))) >> shift
    rescaled = np.where(product < 0, -magnitude, magnitude)
  else:
    rescaled = product

  if p["has_relu"]:
    rescaled = np.maximum(0, rescaled)

  q_x = np.clip(rescaled, -128, 127).astype(np.int8)

q_pred = q_x.argmax(axis=1)
q_acc = accuracy_score(y_test, q_pred)
print(f"float32 Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Quantized int8 Test Accuracy: {q_acc:.4f}")



# Exporting to Hex

In [ ]:
os.makedirs("hex_export", exist_ok=True)

for p in layer_params:
  name = p["name"]

  #export int8 weights - 2-digit two's complement hex, one value per line
  weights_flat = p["q_weights"].T.flatten()
  with open(f"hex_export/{name}_weights.hex", "w") as f:
        for w in weights_flat:
            #& 0xFF converts signed int8 (-127..127) to unsigned 8-bit two's complement
            f.write(f"{int(w) & 0xFF:02x}\n")

  #export int32 biases - 8-digit two's complement hex, one value per line
  biases_flat = p["q_biases"].flatten()
  with open(f"hex_export/{name}_biases.hex", "w") as f:
        for b in biases_flat:
            #& 0xFFFFFFFF converts signed int8 (-127..127) to unsigned 8-bit two's complement
            f.write(f"{int(b) & 0xFFFFFFFF:08x}\n")

  print(f"Exported {name}_weights.hex ({len(weights_flat)} lines) and {name}_biases.hex ({len(biases_flat)} lines)")

#export multipliers and shifts summary
with open("hex_export/quantization_params.txt", "w") as f:
  f.write(f"s_input: {s_input_initial}\n\n")
  for p in layer_params:
      f.write(f"[{p['name']}]\n")
      f.write(f"  multiplier (dec): {p['q_multiplier']}\n")
      f.write(f"  multiplier (hex): 0x{p['q_multiplier']:08x}\n")
      f.write(f"  right_shift:      {p['q_shift']}\n\n")

print("\nExported fixed-point multipliers and shifts to 'hex_export/quantization_params.txt'")